# FairDerm: Skin Lesion Classification

This notebook trains skin lesion classifiers on Fitzpatrick17k, evaluates them on DDI, and analyzes the results.

**Methods:**
- Baseline (cross-entropy + uniform sampling)
- Mixup (cross-entropy + mixup augmentation)
- Reweighted (weighted cross-entropy + class-weighted sampling)
- Focal Loss (gamma=2)
- Proposed (adaptive sampling + mixup)

## Configuration

**Update the paths below for your setup.** Works on Colab, local machine, or any other platform.

In [ ]:
# ============================================================
# SET YOUR PATHS HERE
# ============================================================

# Where the fairderm code package is located
CODE_DIR = '/content/drive/MyDrive/thesis/code'

# Fitzpatrick17k CSV file
FITZPATRICK_CSV = '/content/drive/MyDrive/thesis/data/fitzpatrick17k_processed.csv'

# DDI dataset paths
DDI_IMAGES_DIR = '/content/drive/MyDrive/thesis/data/ddidiversedermatologyimages'
DDI_METADATA = '/content/drive/MyDrive/thesis/data/ddi_metadata.csv'

# Where to save results
RESULTS_DIR = '/content/drive/MyDrive/thesis/results'

# Seeds to train with for reproducibility
SEEDS = [0, 7, 21, 42, 123]

# ============================================================

In [ ]:
# Mount Google Drive if on Colab (comment out if not using Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
except:
    print('Not running on Colab, skipping drive mount')

In [ ]:
# Install dependencies
!pip install timm --quiet

In [ ]:
import sys
sys.path.insert(0, CODE_DIR)

import os
import json
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from pathlib import Path
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from tqdm import tqdm

warnings.filterwarnings('ignore')

from fairderm import (
    bootstrap_ci,
    SkinLesionDataset,
    SkinLesionClassifier,
    DDIDataset,
    create_model,
    get_class_weights,
    get_train_transforms,
    get_eval_transforms,
    get_ddi_eval_transforms,
    run_experiment,
    save_experiment_results,
    compute_metrics,
    compute_fairness_metrics,
)
from fairderm.configs import (
    get_baseline_config,
    get_mixup_config,
    get_reweighted_config,
    get_focal_config,
    get_proposed_config,
    get_groupdro_config,
)

os.makedirs(RESULTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

---
# Part 1: Training
---

## Load and prepare data

In [ ]:
df = pd.read_csv(FITZPATRICK_CSV)

print(f'Total samples: {len(df)}')
print(f'\nLabels:\n{df["label_num"].value_counts()}')
print(f'\nSkin tones:\n{df["tone_group"].value_counts()}')

In [ ]:
# train/val split stratified by label and skin tone
df['stratify_key'] = df['label'].astype(str) + '_' + df['tone_group']

train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['stratify_key']
)

train_df = train_df.drop('stratify_key', axis=1).reset_index(drop=True)
val_df = val_df.drop('stratify_key', axis=1).reset_index(drop=True)

print(f'Train: {len(train_df)}, Val: {len(val_df)}')

In [ ]:
train_transform = get_train_transforms()
eval_transform = get_eval_transforms()

train_dataset = SkinLesionDataset(train_df, transform=train_transform, return_group=True)
val_dataset = SkinLesionDataset(val_df, transform=eval_transform, return_group=True)

class_weights = get_class_weights(train_df, label_col='label_num')
print(f'Class weights: {class_weights}')

## Train all methods

In [ ]:
MODELS = ['baseline', 'mixup', 'reweighted', 'focalloss', 'proposed', 'groupdro']

def get_config(name):
    if name == 'baseline':
        return get_baseline_config()
    elif name == 'mixup':
        return get_mixup_config()
    elif name == 'reweighted':
        return get_reweighted_config(class_weights, device=device)
    elif name == 'focalloss':
        return get_focal_config()
    elif name == 'proposed':
        return get_proposed_config()
    elif name == 'groupdro':
        return get_groupdro_config()
    else:
        raise ValueError(f'Unknown model: {name}')

In [ ]:
# Train all models across all seeds
all_results = {}

SKIP_EXISTING_RUNS = True
REQUIRED_ARTIFACTS = ['model.pt', 'metrics.json', 'training_history.csv']

for model_name in MODELS:
    for seed in SEEDS:
        run_name = f'{model_name}_seed{seed}'
        run_dir = Path(RESULTS_DIR) / run_name

        artifacts_exist = all((run_dir / f).exists() for f in REQUIRED_ARTIFACTS)
        if SKIP_EXISTING_RUNS and artifacts_exist:
            print(f'\nSkipping existing run: {run_name}')
            continue

        if SKIP_EXISTING_RUNS and run_dir.exists() and not artifacts_exist:
            print(f'\nFound incomplete run, retraining: {run_name}')

        print(f'\n{"="*60}')
        print(f'Training: {run_name}')
        print(f'{"="*60}')
        
        config = get_config(model_name)
        results = run_experiment(train_dataset, val_dataset, config, seed=seed, device=device)
        all_results[run_name] = results
        
        save_experiment_results(results, run_name, RESULTS_DIR)
        print(f'Saved to {RESULTS_DIR}/{run_name}')

print(f'\n{"="*60}')
print('Training complete!')
print(f'{"="*60}')

---
# Part 2: DDI Evaluation
---

In [ ]:
# Load DDI dataset
ddi_transforms = get_ddi_eval_transforms()

ddi_dataset = DDIDataset(
    metadata_path=DDI_METADATA,
    images_dir=DDI_IMAGES_DIR,
    transform=ddi_transforms
)

ddi_loader = DataLoader(ddi_dataset, batch_size=32, shuffle=False, num_workers=2)

stats = ddi_dataset.get_statistics()
print(f'DDI samples: {stats["total"]}')
print(f'Per group: {stats["per_group"]}')

In [ ]:
def load_model(model_path):
    model = SkinLesionClassifier(num_classes=2, pretrained=False)
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model = model.to(device)
    model.eval()
    return model

def evaluate_model(model, dataloader):
    model.eval()
    all_preds, all_labels, all_groups, all_probs = [], [], [], []
    
    with torch.no_grad():
        for images, labels, groups in tqdm(dataloader, desc='Evaluating'):
            images = images.to(device)
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_groups.extend(groups.numpy())
            all_probs.extend(probs.cpu().numpy())
    
    return {
        'preds': np.array(all_preds),
        'labels': np.array(all_labels),
        'groups': np.array(all_groups),
        'probs': np.array(all_probs)
    }

In [ ]:
# Evaluate all models on DDI
ddi_results = {model: {} for model in MODELS}

for model_name in MODELS:
    for seed in SEEDS:
        run_name = f'{model_name}_seed{seed}'
        model_path = Path(RESULTS_DIR) / run_name / 'model.pt'
        
        if not model_path.exists():
            print(f'Model not found: {model_path}')
            continue
        
        print(f'\nEvaluating {run_name} on DDI...')
        model = load_model(model_path)
        eval_results = evaluate_model(model, ddi_loader)
        
        overall = compute_metrics(eval_results['labels'], eval_results['preds'], eval_results['probs'])
        fairness = compute_fairness_metrics(eval_results['labels'], eval_results['preds'], eval_results['groups'])
        
        ddi_results[model_name][seed] = {'overall': overall, 'fairness': fairness}
        
        print(f'  Balanced Acc: {overall["balanced_accuracy"]:.4f}, EO Gap: {fairness["equal_opportunity_gap"]:.4f}')
        
        del model
        torch.cuda.empty_cache()

print('\nDDI evaluation complete!')

In [ ]:
# Save DDI results
ddi_dir = Path(RESULTS_DIR) / 'ddi_evaluation'
ddi_dir.mkdir(exist_ok=True)

rows = []
for model_name in MODELS:
    for seed, r in ddi_results[model_name].items():
        rows.append({
            'model': model_name,
            'seed': seed,
            'balanced_accuracy': r['overall']['balanced_accuracy'],
            'roc_auc': r['overall'].get('roc_auc'),
            'equal_opportunity_gap': r['fairness']['equal_opportunity_gap'],
            'tpr_light': r['fairness']['tpr_per_group'].get('Light'),
            'tpr_medium': r['fairness']['tpr_per_group'].get('Medium'),
            'tpr_dark': r['fairness']['tpr_per_group'].get('Dark'),
        })

ddi_df = pd.DataFrame(rows)
ddi_df.to_csv(ddi_dir / 'ddi_results.csv', index=False)
print(f'Saved DDI results to {ddi_dir / "ddi_results.csv"}')

---
# Part 3: Analysis
---

In [ ]:
# Load Fitzpatrick17k results
def load_fitz_results(model, seed):
    path = Path(RESULTS_DIR) / f'{model}_seed{seed}' / 'metrics.json'
    if path.exists():
        with open(path, 'r') as f:
            return json.load(f)
    return None

fitz_results = {}
for model in MODELS:
    fitz_results[model] = {}
    for seed in SEEDS:
        result = load_fitz_results(model, seed)
        if result:
            fitz_results[model][seed] = result

In [ ]:
# Aggregate results using 95% bootstrap confidence intervals
def make_metric(values):
    mean, ci_lower, ci_upper = bootstrap_ci(values)
    return {'mean': mean, 'ci_lower': ci_lower, 'ci_upper': ci_upper, 'values': list(values)}

def aggregate(model_results):
    if not model_results:
        return None

    seeds_data = list(model_results.values())
    result = {'n_seeds': len(seeds_data)}

    # overall metrics
    for key in ['balanced_accuracy', 'roc_auc']:
        values = [s['overall'].get(key) for s in seeds_data if s['overall'].get(key) is not None]
        if values:
            result[key] = make_metric(values)

    # fairness metrics
    for key in ['equal_opportunity_gap']:
        values = [s['fairness'].get(key) for s in seeds_data if s['fairness'].get(key) is not None]
        if values:
            result[key] = make_metric(values)

    # per-group TPR
    for group in ['Light', 'Medium', 'Dark']:
        values = [s['fairness']['tpr_per_group'].get(group) for s in seeds_data
                  if s['fairness']['tpr_per_group'].get(group) is not None]
        if values:
            result[f'tpr_{group.lower()}'] = make_metric(values)

    # worst-group TPR
    worst_values = []
    for s in seeds_data:
        tprs = [s['fairness']['tpr_per_group'].get(g) for g in ['Light', 'Medium', 'Dark']]
        tprs = [t for t in tprs if t is not None]
        if tprs:
            worst_values.append(min(tprs))
    if worst_values:
        result['worst_group_tpr'] = make_metric(worst_values)

    return result

fitz_agg = {m: aggregate(fitz_results[m]) for m in MODELS}
ddi_agg  = {m: aggregate(ddi_results[m])  for m in MODELS}


In [ ]:
# Print summary
def fmt(d):
    if d is None:
        return 'N/A'
    return f"{d['mean']:.4f} [{d['ci_lower']:.4f}, {d['ci_upper']:.4f}]"

print('='*90)
print('FITZPATRICK17K RESULTS  (mean [95% bootstrap CI])')
print('='*90)
print(f'{"Model":<12} {"Balanced Acc":>28} {"Worst-Group TPR":>28} {"EO Gap":>26}')
print('-'*90)
for model in MODELS:
    m = fitz_agg[model]
    if m:
        print(f'{model:<12} {fmt(m.get("balanced_accuracy")):>28}'
              f' {fmt(m.get("worst_group_tpr")):>28} {fmt(m.get("equal_opportunity_gap")):>26}')

print('\n' + '='*90)
print('DDI RESULTS  (mean [95% bootstrap CI])')
print('='*90)
print(f'{"Model":<12} {"Balanced Acc":>28} {"TPR Dark":>28} {"EO Gap":>26}')
print('-'*90)
for model in MODELS:
    m = ddi_agg[model]
    if m:
        print(f'{model:<12} {fmt(m.get("balanced_accuracy")):>28}'
              f' {fmt(m.get("tpr_dark")):>28} {fmt(m.get("equal_opportunity_gap")):>26}')


## Figures

In [ ]:
# Publication-quality figure settings
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['font.family'] = 'sans-serif'

MODEL_COLORS = {
    'baseline': '#7f8c8d',
    'mixup': '#3498db',
    'reweighted': '#9b59b6',
    'focalloss': '#e74c3c',
    'proposed': '#27ae60',
}

MODEL_LABELS = {
    'baseline': 'Baseline',
    'mixup': 'Mixup',
    'reweighted': 'Reweighted',
    'focalloss': 'Focal Loss',
    'proposed': 'Proposed',
}

figures_dir = Path(RESULTS_DIR) / 'figures'
figures_dir.mkdir(exist_ok=True)

In [ ]:
# Figure 1: Balanced accuracy comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(MODELS))
mdata  = [fitz_agg[m]['balanced_accuracy'] for m in MODELS]
values = [d['mean'] for d in mdata]
yerr   = ([d['mean'] - d['ci_lower'] for d in mdata],
           [d['ci_upper'] - d['mean'] for d in mdata])
colors = [MODEL_COLORS[m] for m in MODELS]

bars = ax.bar(x, values, yerr=yerr, color=colors, edgecolor='black',
              linewidth=1.2, capsize=6, error_kw={'linewidth': 2, 'capthick': 2})

# Highlight best model
best_idx = np.argmax(values)
bars[best_idx].set_edgecolor('#27ae60')
bars[best_idx].set_linewidth(3)

ax.set_ylabel('Balanced Accuracy', fontsize=12)
ax.set_title('Overall Classification Performance (Fitzpatrick17k)', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([MODEL_LABELS[m] for m in MODELS], fontsize=11)

y_min = min(d['ci_lower'] for d in mdata) - 0.02
y_max = max(d['ci_upper'] for d in mdata) + 0.02
ax.set_ylim(max(0, y_min), min(1, y_max))

for bar, d in zip(bars, mdata):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (d['ci_upper'] - d['mean']) + 0.005,
            f'{d["mean"]:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig(figures_dir / 'fig1_performance.png')
plt.show()


In [ ]:
# Figure 2: Fairness comparison (EO Gap vs Worst-Group TPR)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

x      = np.arange(len(MODELS))
colors = [MODEL_COLORS[m] for m in MODELS]
labels = [MODEL_LABELS[m] for m in MODELS]

# EO Gap (lower is better)
ax    = axes[0]
mdata = [fitz_agg[m]['equal_opportunity_gap'] for m in MODELS]
values = [d['mean'] for d in mdata]
yerr   = ([d['mean'] - d['ci_lower'] for d in mdata],
           [d['ci_upper'] - d['mean'] for d in mdata])
bars = ax.bar(x, values, yerr=yerr, color=colors, edgecolor='black',
              linewidth=1.2, capsize=6, error_kw={'linewidth': 2, 'capthick': 2})

best_idx = np.argmin(values)
bars[best_idx].set_edgecolor('#e74c3c')
bars[best_idx].set_linewidth(3)

ax.set_ylabel('Equal Opportunity Gap (lower is better)', fontsize=11)
ax.set_title('(a) Gap-Based Fairness', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)

for bar, d in zip(bars, mdata):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (d['ci_upper'] - d['mean']) + 0.002,
            f'{d["mean"]:.3f}', ha='center', va='bottom', fontsize=9)

# Worst-Group TPR (higher is better)
ax    = axes[1]
mdata = [fitz_agg[m]['worst_group_tpr'] for m in MODELS]
values = [d['mean'] for d in mdata]
yerr   = ([d['mean'] - d['ci_lower'] for d in mdata],
           [d['ci_upper'] - d['mean'] for d in mdata])
bars = ax.bar(x, values, yerr=yerr, color=colors, edgecolor='black',
              linewidth=1.2, capsize=6, error_kw={'linewidth': 2, 'capthick': 2})

best_idx = np.argmax(values)
bars[best_idx].set_edgecolor('#27ae60')
bars[best_idx].set_linewidth(3)

ax.set_ylabel('Worst-Group TPR (higher is better)', fontsize=11)
ax.set_title('(b) Minimax Fairness', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=10)

for bar, d in zip(bars, mdata):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (d['ci_upper'] - d['mean']) + 0.005,
            f'{d["mean"]:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(figures_dir / 'fig2_fairness.png')
plt.show()


In [ ]:
# Figure 3: Per-group sensitivity
fig, ax = plt.subplots(figsize=(12, 7))

x           = np.arange(len(MODELS))
width       = 0.25
skin_colors = ['#F5DEB3', '#CD853F', '#4A3728']

for i, (group, color) in enumerate(zip(['light', 'medium', 'dark'], skin_colors)):
    mdata  = [fitz_agg[m][f'tpr_{group}'] for m in MODELS]
    values = [d['mean'] for d in mdata]
    yerr   = ([d['mean'] - d['ci_lower'] for d in mdata],
               [d['ci_upper'] - d['mean'] for d in mdata])
    ax.bar(x + (i-1)*width, values, width, yerr=yerr, label=group.capitalize(),
           color=color, edgecolor='black', linewidth=1, capsize=4,
           error_kw={'linewidth': 1.5, 'capthick': 1.5})

ax.set_ylabel('True Positive Rate (Sensitivity)', fontsize=12)
ax.set_title('Per-Group Cancer Detection Sensitivity by Skin Tone', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([MODEL_LABELS[m] for m in MODELS], fontsize=11)
ax.legend(title='Skin Tone', fontsize=10)

all_values = [fitz_agg[m][f'tpr_{g}']['mean'] for m in MODELS for g in ['light', 'medium', 'dark']]
ax.set_ylim(min(all_values) - 0.05, max(all_values) + 0.05)

plt.tight_layout()
plt.savefig(figures_dir / 'fig3_per_group_tpr.png')
plt.show()


In [ ]:
# Figure 4: Accuracy vs fairness scatter
fig, ax = plt.subplots(figsize=(10, 8))

markers = ['o', 's', '^', 'D', 'P']

for i, model in enumerate(MODELS):
    m  = fitz_agg[model]
    wg = m['worst_group_tpr']
    ba = m['balanced_accuracy']
    ax.errorbar(
        wg['mean'], ba['mean'],
        xerr=[[wg['mean'] - wg['ci_lower']], [wg['ci_upper'] - wg['mean']]],
        yerr=[[ba['mean'] - ba['ci_lower']], [ba['ci_upper'] - ba['mean']]],
        fmt=markers[i], markersize=14, color=MODEL_COLORS[model],
        label=MODEL_LABELS[model], capsize=5, markeredgecolor='black',
        markeredgewidth=1.5, elinewidth=2
    )

ax.set_xlabel('Worst-Group TPR (higher is better)', fontsize=12)
ax.set_ylabel('Balanced Accuracy (higher is better)', fontsize=12)
ax.set_title('Accuracy vs Fairness Trade-off', fontsize=14, fontweight='bold')
ax.legend(loc='lower left', fontsize=11)

ax.annotate('Better', xy=(0.98, 0.98), xycoords='axes fraction',
            fontsize=10, ha='right', va='top', style='italic', color='gray')

plt.tight_layout()
plt.savefig(figures_dir / 'fig4_tradeoff.png')
plt.show()


In [ ]:
# Figure 5: DDI External Evaluation
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# DDI Balanced Accuracy
ax = axes[0]
avail = [m for m in MODELS if ddi_agg[m]]
mdata  = [ddi_agg[m]['balanced_accuracy'] for m in avail]
values = [d['mean'] for d in mdata]
yerr   = ([d['mean'] - d['ci_lower'] for d in mdata],
           [d['ci_upper'] - d['mean'] for d in mdata])
x_a    = np.arange(len(avail))
colors_a = [MODEL_COLORS[m] for m in avail]
labels_a = [MODEL_LABELS[m] for m in avail]

bars = ax.bar(x_a, values, yerr=yerr, color=colors_a, edgecolor='black',
              linewidth=1.2, capsize=6, error_kw={'linewidth': 2, 'capthick': 2})

best_idx = np.argmax(values)
bars[best_idx].set_edgecolor('#27ae60')
bars[best_idx].set_linewidth(3)

ax.set_ylabel('Balanced Accuracy', fontsize=11)
ax.set_title('(a) DDI Performance', fontsize=13, fontweight='bold')
ax.set_xticks(x_a)
ax.set_xticklabels(labels_a, fontsize=10)

for bar, d in zip(bars, mdata):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (d['ci_upper'] - d['mean']) + 0.005,
            f'{d["mean"]:.3f}', ha='center', va='bottom', fontsize=9)

# DDI TPR Dark
ax = axes[1]
avail_d = [m for m in MODELS if ddi_agg[m] and ddi_agg[m].get('tpr_dark')]
mdata_d  = [ddi_agg[m]['tpr_dark'] for m in avail_d]
values_d = [d['mean'] for d in mdata_d]
yerr_d   = ([d['mean'] - d['ci_lower'] for d in mdata_d],
             [d['ci_upper'] - d['mean'] for d in mdata_d])
x_d      = np.arange(len(avail_d))
colors_d = [MODEL_COLORS[m] for m in avail_d]
labels_d = [MODEL_LABELS[m] for m in avail_d]

bars = ax.bar(x_d, values_d, yerr=yerr_d, color=colors_d, edgecolor='black',
              linewidth=1.2, capsize=6, error_kw={'linewidth': 2, 'capthick': 2})

best_idx = np.argmax(values_d)
bars[best_idx].set_edgecolor('#27ae60')
bars[best_idx].set_linewidth(3)

ax.set_ylabel('TPR (Dark Skin)', fontsize=11)
ax.set_title('(b) DDI Fairness (Dark Skin Sensitivity)', fontsize=13, fontweight='bold')
ax.set_xticks(x_d)
ax.set_xticklabels(labels_d, fontsize=10)

for bar, d in zip(bars, mdata_d):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + (d['ci_upper'] - d['mean']) + 0.005,
            f'{d["mean"]:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(figures_dir / 'fig5_ddi_evaluation.png')
plt.show()


In [ ]:
# Save results table with ROC-AUC
rows = []
for model in MODELS:
    m = fitz_agg[model]
    d = ddi_agg[model]
    rows.append({
        'Model': MODEL_LABELS[model],
        # Fitzpatrick17k results
        'Fitz Balanced Acc': fmt(m.get('balanced_accuracy')),
        'Fitz ROC-AUC': fmt(m.get('roc_auc')),
        'Fitz Worst-Group TPR': fmt(m.get('worst_group_tpr')),
        'Fitz EO Gap': fmt(m.get('equal_opportunity_gap')),
        # DDI results
        'DDI Balanced Acc': fmt(d.get('balanced_accuracy')) if d else 'N/A',
        'DDI TPR Dark': fmt(d.get('tpr_dark')) if d else 'N/A',
        'DDI EO Gap': fmt(d.get('equal_opportunity_gap')) if d else 'N/A',
    })

results_df = pd.DataFrame(rows)
results_df.to_csv(Path(RESULTS_DIR) / 'results_summary.csv', index=False)

# Print formatted tables
print('='*100)
print('COMPLETE RESULTS SUMMARY')
print('='*100)
print('\nFitzpatrick17k Results:')
print(results_df[['Model', 'Fitz Balanced Acc', 'Fitz ROC-AUC', 'Fitz Worst-Group TPR', 'Fitz EO Gap']].to_string(index=False))
print('\nDDI External Evaluation Results:')
print(results_df[['Model', 'DDI Balanced Acc', 'DDI TPR Dark', 'DDI EO Gap']].to_string(index=False))
print(f'\nResults saved to {Path(RESULTS_DIR) / "results_summary.csv"}')

---
## Done!

Models saved to the results directory. Figures saved to the figures subdirectory.